In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!rm -rf /content/Parking
!git clone https://github.com/hou1020/Parking.git /content/Parking
%cd /content/Parking/parking-lot-mapping-tool
!ls

In [ ]:
pwd

In [ ]:
!pip install -q "pytorch-lightning<2.0.0"
!pip install -q "transformers==4.40.2" datasets roboflow
!pip install -q evaluate rasterio geojson imageio geopandas

# Leeds TIF Batch Inference with UK Removal

逐个处理输入 TIF 文件夹里的 GeoTIFF，原始模型结果保存到 `output_files/<prefix>_original`，UK buildings 和 roads 扣除后的结果保存到 `output_files/<prefix>_removal`。例如 `files/leeds_tif` 会输出到 `output_files/leeds_original` 和 `output_files/leeds_removal`。

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import shutil
import warnings

import geopandas as gpd
import imageio.v2 as iio
import numpy as np
import pandas as pd
import rasterio
from rasterio.features import shapes
import torch
from PIL import Image
from shapely.geometry import shape
from shapely.ops import unary_union
from torch import nn
from torch.utils.data import DataLoader
from tqdm import tqdm
from transformers import SegformerFeatureExtractor
from huggingface_hub import hf_hub_download

from functions import convert_to_rgb, split_images
from inference import SemanticSegmentationDataset, SegformerFinetuner
from post_processing_uk import postprocess_prediction_uk

warnings.filterwarnings("ignore")

In [ ]:
TIF_DIR = Path("files/leeds_tif")
OUTPUT_ROOT = Path("output_files")
OUTPUT_PREFIX = TIF_DIR.name[:-4] if TIF_DIR.name.endswith("_tif") else TIF_DIR.name
ORIGINAL_OUTPUT_DIR = OUTPUT_ROOT / f"{OUTPUT_PREFIX}_original"
REMOVAL_OUTPUT_DIR = OUTPUT_ROOT / f"{OUTPUT_PREFIX}_removal"
RGB_PATH = Path("files/large_img.PNG")
MODEL_PATH = Path("files/best_model.ckpt")

BATCH_SIZE = 5
NUM_WORKERS = 4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ORIGINAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REMOVAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Device:", DEVICE)
print("TIF count:", len(sorted(TIF_DIR.glob("*.tif"))))
print("Original output:", ORIGINAL_OUTPUT_DIR)
print("Removal output:", REMOVAL_OUTPUT_DIR)


In [ ]:
# 如果 files/best_model.ckpt 不存在，就从 Hugging Face 下载。
# 本地第一次运行这一步需要联网。
if not MODEL_PATH.exists() or MODEL_PATH.stat().st_size < 1024:
    cached_file = hf_hub_download(
        repo_id="UTEL-UIUC/SegFormer-large-parking",
        filename="best_model.ckpt",
    )
    shutil.copy(cached_file, MODEL_PATH)

print("Model checkpoint:", MODEL_PATH)

In [ ]:
# 加载 SegFormer 模型。这里保留 main.ipynb 的模型结构，但支持 CPU/GPU 自动选择。
feature_extractor = SegformerFeatureExtractor.from_pretrained(
    "nvidia/segformer-b5-finetuned-cityscapes-1024-1024"
)
feature_extractor.do_reduce_labels = False
feature_extractor.size = 512

# 先用一个 tif 初始化 small_images，从而创建 dataset 和 id2label。
first_tif = sorted(TIF_DIR.glob("*.tif"))[0]
convert_to_rgb(first_tif, RGB_PATH)
first_img = iio.imread(RGB_PATH)
split_images(first_img, num=0)

test_dataset = SemanticSegmentationDataset("small_images/", feature_extractor)
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)

segformer_finetuner = SegformerFinetuner(
    test_dataset.id2label,
    train_dataloader=test_dataloader,
    val_dataloader=test_dataloader,
    test_dataloader=test_dataloader,
    metrics_interval=10,
).load_from_checkpoint(MODEL_PATH, id2label=test_dataset.id2label)

segformer_finetuner.model.to(DEVICE)
segformer_finetuner.model.eval()
print("Model loaded")

In [ ]:
def read_raster_metadata(tif_path):
    """读取 tif 的真实 transform 和 CRS，用于 geospatial raster-to-vector。"""
    with rasterio.open(tif_path) as dataset:
        return dataset.height, dataset.width, dataset.transform, dataset.crs


def fix_geometry(geom):
    """修复模型 mask 转 polygon 后可能出现的无效几何。"""
    if geom is None or geom.is_empty:
        return None

    try:
        from shapely.validation import make_valid
        geom = make_valid(geom)
    except Exception:
        try:
            geom = geom.buffer(0)
        except Exception:
            return None

    if geom is None or geom.is_empty:
        return None
    return geom


def safe_difference(geom, inner_union):
    """扣除内部洞；如果拓扑失败，就修复后重试，避免整张图中断。"""
    geom = fix_geometry(geom)
    if geom is None:
        return None

    try:
        return fix_geometry(geom.difference(inner_union))
    except Exception:
        try:
            return fix_geometry(geom.buffer(0).difference(inner_union.buffer(0)))
        except Exception:
            return geom


def save_original_geojson(polygons, inner_polygons, output_path, crs):
    """保存原始 polygon。CRS 使用输入 tif 的 CRS。"""
    output_path.parent.mkdir(parents=True, exist_ok=True)

    polygons = [geom for geom in (fix_geometry(g) for g in polygons) if geom is not None]
    inner_polygons = [geom for geom in (fix_geometry(g) for g in inner_polygons) if geom is not None]

    outer_gdf = gpd.GeoDataFrame({"geometry": polygons}, geometry="geometry", crs=crs)
    if inner_polygons and not outer_gdf.empty:
        inner_union = fix_geometry(unary_union(inner_polygons))
        if inner_union is not None:
            outer_gdf["geometry"] = outer_gdf.geometry.apply(lambda geom: safe_difference(geom, inner_union))

    outer_gdf = outer_gdf[outer_gdf.geometry.notna() & ~outer_gdf.geometry.is_empty]
    outer_gdf.to_file(output_path, driver="GeoJSON")


def vectorize_mask_to_geojson(seg_t, raster_height, raster_width, transform, crs, output_path):
    """用 rasterio.features.shapes 按 tif transform 直接矢量化预测 mask。"""
    output_path.parent.mkdir(parents=True, exist_ok=True)

    mask = (seg_t[:raster_height, :raster_width] == 1).astype(np.uint8)
    geoms = []

    for geom_mapping, value in shapes(mask, mask=mask.astype(bool), transform=transform):
        if value != 1:
            continue
        geom = fix_geometry(shape(geom_mapping))
        if geom is not None:
            geoms.append(geom)

    gdf = gpd.GeoDataFrame({"geometry": geoms}, geometry="geometry", crs=crs)
    gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty]
    gdf = gdf[gdf.geometry.geom_type.isin(["Polygon", "MultiPolygon"])]
    gdf.to_file(output_path, driver="GeoJSON")


def process_batch(batch):
    images = batch["pixel_values"].to(DEVICE)
    masks = batch["labels"].to(DEVICE)

    with torch.no_grad():
        outputs = segformer_finetuner.model(images, masks)
        logits = outputs[1]
        logits = nn.functional.interpolate(
            logits,
            size=masks.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )
        predicted_mask = logits.argmax(dim=1).cpu().numpy()

    return predicted_mask


def save_mask(img_number, mask, dataset):
    img_name = dataset.imgs[img_number].split(".")[0]
    Image.fromarray(mask.astype(np.uint8)).save(
        "small_images/Masks/" + img_name + ".PNG",
        "PNG",
        quality=100,
    )


def predict_current_small_images(tile_count, padded_height, padded_width):
    """按 main.ipynb 的方式：预测每个 tile，保存 mask，再读回拼接。"""
    dataset = SemanticSegmentationDataset("small_images/", feature_extractor)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)

    n_cols = padded_width // 512
    seg_t = np.zeros((padded_height, padded_width))

    save_workers = max(1, NUM_WORKERS)
    with ThreadPoolExecutor(max_workers=save_workers) as executor:
        futures = []
        for batch_idx, batch in enumerate(tqdm(dataloader, desc="Processing images")):
            masks = process_batch(batch)
            for i in range(masks.shape[0]):
                img_number = batch_idx * BATCH_SIZE + i
                if img_number < tile_count:
                    futures.append(executor.submit(save_mask, img_number, masks[i], dataset))

        for future in futures:
            future.result()

    for img_number in range(tile_count):
        row = img_number // n_cols
        col = img_number % n_cols
        pred = Image.open("small_images/Masks/" + str(img_number + 1) + ".PNG")
        seg_t[row * 512:(row + 1) * 512, col * 512:(col + 1) * 512] = pred

    return seg_t


In [ ]:
def process_tif(tif_path):
    """处理单个 tif，并分别输出 raw 和 UK buildings/roads removal 后的 GeoJSON。"""
    name = tif_path.stem
    raw_output_path = ORIGINAL_OUTPUT_DIR / f"{name}_original.geojson"
    removal_output_path = REMOVAL_OUTPUT_DIR / f"{name}_removal.geojson"
    print(f"\nProcessing {name}")

    raster_height, raster_width, transform, crs = read_raster_metadata(tif_path)

    convert_to_rgb(tif_path, RGB_PATH)
    img = iio.imread(RGB_PATH)
    tile_count, padded_height, padded_width = split_images(img, num=0)

    seg_t = predict_current_small_images(tile_count, padded_height, padded_width)
    vectorize_mask_to_geojson(seg_t, raster_height, raster_width, transform, crs, raw_output_path)
    postprocess_prediction_uk(raw_output_path, tif_path, removal_output_path)

    print(f"Saved raw: {raw_output_path}")
    print(f"Saved removal: {removal_output_path}")
    return raw_output_path, removal_output_path


In [ ]:
tif_paths = sorted(TIF_DIR.glob("*.tif"))
outputs = []

for tif_path in tif_paths:
    raw_path, removal_path = process_tif(tif_path)
    outputs.append({
        "tif": str(tif_path),
        "original_geojson": str(raw_path),
        "removal_geojson": str(removal_path),
    })

pd.DataFrame(outputs)
